# 🏦 Banking AI-Agent: Google Colab

This notebook sets up the Banking AI-Agent on Google Colab.
1. **Ollama Backend**
2. **FastAPI API Server**
3. **Streamlit Web Frontend**

### Step 1: Clone or Update the Repository

In [ ]:
!git clone https://github.com/TheWallOnFire/NLP-Project2.git || (cd NLP-Project2 && git pull)
%cd NLP-Project2/Ex3

### Step 2: Install and Start Ollama

In [ ]:
!apt-get update -qq && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

import os
import subprocess
import threading
import time

env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0"

def run_ollama_serve():
    subprocess.Popen(["ollama", "serve"], env=env)

threading.Thread(target=run_ollama_serve, daemon=True).start()
time.sleep(5)

### Step 3: Pull the Model

In [ ]:
!ollama pull gpt-oss:20b

### Step 4: Install Dependencies

In [ ]:
!pip install -r requirements.txt
!pip install streamlit streamlit_mic_recorder

### Step 5: Expose Ports via Cloudflare (Stable)
We use **cloudflared** because it is significantly more stable than Localtunnel for Streamlit apps.

In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

import subprocess
import threading
import time
import re

def start_tunnel(port):
    def log_output(process, port):
        for line in iter(process.stdout.readline, b''):
            line = line.decode('utf-8')
            if 'trycloudflare.com' in line:
                url = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
                if url:
                    print(f"\n✅ Port {port} is live at: {url.group(0)}")

    process = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://localhost:{port}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    threading.Thread(target=log_output, args=(process, port), daemon=True).start()

print("Starting tunnels...")
start_tunnel(8000) # API
start_tunnel(8501) # Frontend
time.sleep(5)

### Step 6: Launch

In [ ]:
import subprocess
subprocess.Popen(["python", "run.py"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
!streamlit run frontend/app.py --server.port 8501 --server.enableCORS false --server.enableXsrfProtection false